# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 5 — Vector stores y recuperación

ITBA Educación Ejecutiva

Construcción del motor de búsqueda semántica: los chunks embebidos de la Clase 4 necesitan un lugar donde vivir y buscarse — eso es un vector store.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo. No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 10 (Laboratorio)**: checkpoint del proyecto integrador.

## Objetivo de la clase

Al final de la clase vas a poder:
- Explicar qué automatiza un vector store que una lista de vectores en memoria no.
- Construir y consultar un índice con ChromaDB, incluyendo filtros por metadata.
- Persistir un índice y recuperarlo en otra sesión.
- Comparar ChromaDB contra FAISS y elegir con criterio.
- Diagnosticar manualmente resultados irrelevantes o redundantes en una búsqueda semántica.

**Checkpoint del proyecto integrador de hoy**: índice vectorial construido y buscador semántico funcionando (ver `proyecto-integrador/README.md`, checkpoint Clase 5).

## Agenda

1. Almacenamiento vectorial y recuperación por vecinos cercanos
2. ChromaDB: índice, agregar, consultar (top-k)
3. Filtrado por metadata
4. Persistencia del índice
5. Comparación con FAISS
6. Selección de modelos de embeddings
7. Diagnóstico de resultados irrelevantes y redundantes
8. Embeddings locales
9. Evaluación manual de consultas
10. Laboratorio: buscador semántico

## 0. Punto de partida

Recordá de la Clase 4: `embed_bedrock`, `limpiar_texto`, chunking con metadata. Hoy partimos de un corpus chico de 3 "documentos" ya chunkeados, para enfocarnos en el índice y la búsqueda, no en la ingesta otra vez.

In [2]:
import os, json, re, unicodedata, hashlib
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos: list[str], input_type: str = "search_document") -> list[list[float]]:
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    resp_body = json.loads(resp["body"].read())
    return resp_body["embeddings"]["float"]

In [3]:
# Corpus de juguete: 3 "documentos" cortos, ya separados en chunks con metadata de tema.
corpus = [
    {"id": "apis-001", "texto": "La Converse API de Bedrock unifica la forma de llamar a distintos modelos: cambia el modelId, no el código.", "tema": "apis", "doc": "guia-apis"},
    {"id": "apis-002", "texto": "Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.", "tema": "apis", "doc": "guia-apis"},
    {"id": "seguridad-001", "texto": "La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.", "tema": "seguridad", "doc": "guia-seguridad"},
    {"id": "seguridad-002", "texto": "El principio de mínimo privilegio limita qué herramientas puede invocar un agente según el contexto.", "tema": "seguridad", "doc": "guia-seguridad"},
    {"id": "rag-001", "texto": "Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.", "tema": "rag", "doc": "guia-rag"},
    {"id": "rag-002", "texto": "El top-k determina cuántos fragmentos recuperados se incluyen como contexto para el modelo generador.", "tema": "rag", "doc": "guia-rag"},
]

vectores = embed_bedrock([c["texto"] for c in corpus])
print(f"{len(vectores)} vectores de dimensión {len(vectores[0])}")

6 vectores de dimensión 1536


## 1. Almacenamiento vectorial y recuperación por vecinos cercanos

Con 6 vectores, buscar el más parecido a una consulta es trivial: calcular similitud coseno contra cada uno y ordenar. Con 6 millones, esa búsqueda lineal (fuerza bruta) no escala — un **vector store** indexa los vectores (estructuras tipo HNSW, IVF) para encontrar los vecinos más cercanos sin comparar contra todos.

Primero, la versión "a mano" (fuerza bruta), para entender qué es lo que el vector store va a automatizar:

In [5]:
import numpy as np

def similitud_coseno(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def buscar_fuerza_bruta(consulta_vec, vectores, k=3):
    scores = [similitud_coseno(consulta_vec, v) for v in vectores]
    indices_ordenados = np.argsort(scores)[::-1][:k]
    return [(corpus[i]["id"], scores[i]) for i in indices_ordenados]

consulta = "¿Cómo se controla la temperatura de generación?"
consulta_vec = embed_bedrock([consulta], input_type="search_query")[0]

for id_chunk, score in buscar_fuerza_bruta(consulta_vec, vectores, k=3):
    print(f"{id_chunk}  (sim={score:.3f})")

apis-002  (sim=0.230)
rag-002  (sim=0.182)
rag-001  (sim=0.159)


Notá `input_type="search_query"` (distinto de `"search_document"` usado para el corpus) — varios modelos de embeddings, incluido Cohere Embed, usan un modo distinto internamente según si el texto es la consulta o el documento a indexar. No es un detalle cosmético: usar `search_document` para una consulta puede degradar la calidad de la búsqueda.

## 2. ChromaDB: índice, agregar, consultar

ChromaDB es un vector store embebido (corre en el mismo proceso, sin servidor aparte) con metadata y persistencia incorporadas — el punto de entrada más simple para RAG.

In [7]:
# %pip install chromadb

import chromadb

cliente_chroma = chromadb.PersistentClient(path="./chroma_data")
coleccion = cliente_chroma.get_or_create_collection("corpus_curso")

coleccion.add(
    ids=[c["id"] for c in corpus],
    embeddings=vectores,
    documents=[c["texto"] for c in corpus],
    metadatas=[{"tema": c["tema"], "doc": c["doc"]} for c in corpus],
)

print("Chunks en la colección:", coleccion.count())

Chunks en la colección: 6


In [8]:
resultado = coleccion.query(query_embeddings=[consulta_vec], n_results=3)

for id_, texto, dist in zip(resultado["ids"][0], resultado["documents"][0], resultado["distances"][0]):
    print(f"{id_}  (distancia={dist:.3f})  {texto[:70]}...")

apis-002  (distancia=1.541)  Los parámetros de inferencia como temperature y maxTokens van en infer...
rag-002  (distancia=1.636)  El top-k determina cuántos fragmentos recuperados se incluyen como con...
rag-001  (distancia=1.683)  Un pipeline RAG combina recuperación de evidencia con generación suste...


Ojo: Chroma devuelve **distancia**, no similitud — con la métrica default (L2), valores más chicos = más parecido (al revés que la similitud coseno, donde más alto = más parecido). Revisá qué métrica usa tu colección antes de interpretar los números.

**🏠 Practicá**: corré la misma consulta con `n_results=6` (todos los chunks) y confirmá que el orden coincide con el de `buscar_fuerza_bruta` de la sección 1 — dos implementaciones distintas del mismo concepto deberían dar el mismo ranking.

## 3. Filtrado por metadata

Además de similitud semántica, casi siempre hace falta filtrar por un campo exacto — por ejemplo, restringir la búsqueda a un tema o una fuente específica.

In [10]:
resultado_filtrado = coleccion.query(
    query_embeddings=[consulta_vec],
    n_results=3,
    where={"tema": {"$in": ["seguridad", "rag"]}},  # solo busca entre los chunks con tema="apis"
)

for id_, texto in zip(resultado_filtrado["ids"][0], resultado_filtrado["documents"][0]):
    print(f"{id_}  {texto[:70]}...")

rag-002  El top-k determina cuántos fragmentos recuperados se incluyen como con...
rag-001  Un pipeline RAG combina recuperación de evidencia con generación suste...
seguridad-001  La inyección de prompts ocurre cuando contenido no confiable manipula ...


**🏠 Practicá**: probá el filtro `where={"tema": {"$in": ["seguridad", "rag"]}}` con la misma consulta (que es sobre temperatura/APIs) y mirá qué devuelve — como no hay chunks de ese tema relevantes, es un buen ejemplo de "resultados forzados" que vamos a diagnosticar en la sección 7.

## 4. Persistencia del índice

Como usamos `PersistentClient`, los datos ya están en disco en `./chroma_data` — no hace falta "guardar" explícitamente. Confirmemos abriendo una conexión nueva.

In [11]:
cliente_nuevo = chromadb.PersistentClient(path="./chroma_data")
coleccion_reabierta = cliente_nuevo.get_collection("corpus_curso")
print("Chunks recuperados tras reabrir:", coleccion_reabierta.count())

Chunks recuperados tras reabrir: 6


Esto es clave para la separación ingesta/consulta de la Clase 4: la ingesta corre una vez y persiste; la consulta (en producción, probablemente en un proceso distinto — una API) solo necesita abrir el índice ya construido, no reconstruirlo.

## 5. Comparación con FAISS

FAISS (Facebook AI Similarity Search) es una librería de bajo nivel: ultrarrápida para operaciones vectoriales puras, pero **sin metadata ni persistencia incorporada** — eso hay que armarlo a mano.

In [13]:
# %pip install faiss-cpu

import faiss
import numpy as np

matriz = np.array(vectores, dtype="float32")
faiss.normalize_L2(matriz)  # para que el producto interno equivalga a similitud coseno

indice_faiss = faiss.IndexFlatIP(matriz.shape[1])
indice_faiss.add(matriz)

consulta_matriz = np.array([consulta_vec], dtype="float32")
faiss.normalize_L2(consulta_matriz)

similitudes, posiciones = indice_faiss.search(consulta_matriz, k=3)
for pos, sim in zip(posiciones[0], similitudes[0]):
    print(f"{pos,corpus[pos]['id']}  (sim={sim:.3f})")

(np.int64(1), 'apis-002')  (sim=0.230)
(np.int64(5), 'rag-002')  (sim=0.182)
(np.int64(4), 'rag-001')  (sim=0.159)


In [22]:
a = np.random.randn(1000000, 1536)

In [24]:

import faiss
import numpy as np

matriz = np.array(a, dtype="float32")
faiss.normalize_L2(matriz)  # para que el producto interno equivalga a similitud coseno

indice_faiss = faiss.IndexFlatIP(matriz.shape[1])
indice_faiss.add(matriz)

consulta_matriz = np.array([consulta_vec], dtype="float32")
faiss.normalize_L2(consulta_matriz)

similitudes, posiciones = indice_faiss.search(consulta_matriz, k=3)
for pos, sim in zip(posiciones[0], similitudes[0]):
    print(f"{pos}  (sim={sim:.3f})")

47863  (sim=0.129)
651874  (sim=0.119)
717589  (sim=0.118)


Notá que FAISS devuelve **posiciones** (índices del array), no los ids ni la metadata — esa correspondencia (`posición → chunk_id/metadata`) es responsabilidad tuya mantenerla sincronizada (típicamente con una lista o dict paralelo). Chroma hace esto automáticamente; a cambio, es más lento en escalas muy grandes (decenas de millones de vectores).

| | ChromaDB | FAISS |
|---|---|---|
| Metadata y filtros | Incorporado | Manual |
| Persistencia | Incorporada (`PersistentClient`) | Manual (`faiss.write_index`/`read_index`) |
| Performance a gran escala | Buena | Mejor (es el estándar de facto para escala masiva) |
| Curva de entrada | Baja | Media — hay que armar el resto del sistema alrededor |

Para el TP integrador, con un corpus de documentación técnica (no millones de documentos), ChromaDB alcanza y sobra — FAISS se justifica cuando el volumen o la latencia lo requieren.

## 6. Selección de modelos de embeddings

Ya lo vimos como decisión de proveedor (Clase 1 y 4) — en el contexto de un vector store, dos cosas más a tener en cuenta:

- **Dimensión del vector**: más dimensiones → más precisión potencial, pero más memoria/disco y búsquedas más lentas. Cohere Embed 4 permite elegir entre 256 y 1536 — no asumas que "más grande es mejor" sin medir.
- **Consistencia**: si cambiás de modelo de embeddings, **hay que re-embeber y re-indexar todo el corpus** — los vectores de dos modelos distintos no son comparables entre sí (ya lo vimos en la Clase 4, vale la pena repetirlo: es el error más común y más silencioso en producción).

Para comparar modelos de embeddings de forma objetiva, el benchmark de referencia de la industria es **MTEB** (Massive Text Embedding Benchmark) — vale la pena consultarlo antes de elegir uno para un proyecto real, en vez de ir por default/popularidad.

## 7. Diagnóstico de resultados irrelevantes y redundantes

Dos problemas típicos de un buscador semántico recién armado:

- **Resultados irrelevantes**: el top-k siempre devuelve *algo*, incluso si nada del corpus es realmente relevante — la similitud coseno nunca es "cero", solo más baja. Hay que mirar el *valor* de la similitud, no solo confiar en que "está en el top-k".
- **Resultados redundantes**: varios chunks casi idénticos (por overlap de chunking, o porque el documento repite la misma idea) ocupan varios lugares del top-k, desplazando resultados distintos pero relevantes.

In [ ]:
# Irrelevante: una consulta sin relación real con el corpus
consulta_irrelevante = "¿Cuál es la receta de una tarta de manzana?"
vec_irrelevante = embed_bedrock([consulta_irrelevante], input_type="search_query")[0]

print("Consulta irrelevante al corpus:")
for id_chunk, score in buscar_fuerza_bruta(vec_irrelevante, vectores, k=3):
    print(f"  {id_chunk}  (sim={score:.3f})")
print("-> Notá que la similitud máxima es mucho más baja que en una consulta relevante (sección 1).")

In [ ]:
# Redundante: agregamos un chunk casi idéntico a uno existente
chunk_redundante = "Los parámetros como la temperatura y el máximo de tokens se configuran en inferenceConfig."
vec_redundante = embed_bedrock([chunk_redundante], input_type="search_document")[0]

vectores_con_redundante = vectores + [vec_redundante]
corpus_con_redundante = corpus + [{"id": "apis-003-redundante", "texto": chunk_redundante}]

for id_chunk, score in [(corpus_con_redundante[i]["id"], s)
                        for i, s in sorted(enumerate([similitud_coseno(consulta_vec, v) for v in vectores_con_redundante]),
                                            key=lambda x: -x[1])[:3]]:
    print(f"{id_chunk}  (sim={score:.3f})")
print("-> apis-002 y apis-003-redundante compiten por el mismo lugar del top-k con contenido casi igual.")

**🏠 Practicá**: definí un umbral mínimo de similitud (ej. 0.3) y modificá `buscar_fuerza_bruta` para que descarte resultados por debajo de ese umbral en vez de devolver siempre k resultados — así una consulta irrelevante puede devolver "no encontré nada relevante" en vez de forzar 3 resultados de baja calidad.

## 8. Embeddings locales

Recordá `embed_ollama` de la Clase 4 (`nomic-embed-text`, vía Ollama). Para el vector store, el flujo es idéntico — Chroma y FAISS no saben ni les importa de dónde vino el vector, solo reciben arrays de floats. La diferencia está en costo (gratis, sin llamadas a una API externa) y privacidad (nada sale de tu máquina) a cambio de, generalmente, algo menos de calidad que los modelos propietarios más grandes.

## 9. Evaluación manual de consultas

Antes de automatizar métricas (Clase 8), vale la pena simplemente *mirar* los resultados de un puñado de consultas representativas — es sorprendente cuántos problemas se detectan a ojo.

In [26]:
corpus

[{'id': 'apis-001',
  'texto': 'La Converse API de Bedrock unifica la forma de llamar a distintos modelos: cambia el modelId, no el código.',
  'tema': 'apis',
  'doc': 'guia-apis'},
 {'id': 'apis-002',
  'texto': 'Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.',
  'tema': 'apis',
  'doc': 'guia-apis'},
 {'id': 'seguridad-001',
  'texto': 'La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.',
  'tema': 'seguridad',
  'doc': 'guia-seguridad'},
 {'id': 'seguridad-002',
  'texto': 'El principio de mínimo privilegio limita qué herramientas puede invocar un agente según el contexto.',
  'tema': 'seguridad',
  'doc': 'guia-seguridad'},
 {'id': 'rag-001',
  'texto': 'Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.',
  'tema': 'rag',
  'doc': 'guia-rag'},
 {'id': 'rag-002',
  'texto': 'El top-k determina cuántos fragmentos recuperados se incluyen como cont

In [25]:
consultas_de_prueba = [
    "¿Qué controla la temperatura en una llamada al modelo?",
    "¿Qué es la inyección de prompts?",
    "¿Para qué sirve el top-k en RAG?",
    "¿Cómo se hace un backup de la base de datos?",  # fuera de dominio, a propósito
]

for q in consultas_de_prueba:
    vec_q = embed_bedrock([q], input_type="search_query")[0]
    top1_id, top1_score = buscar_fuerza_bruta(vec_q, vectores, k=1)[0]
    print(f"Q: {q}\n  -> {top1_id} (sim={top1_score:.3f})\n")

Q: ¿Qué controla la temperatura en una llamada al modelo?
  -> apis-001 (sim=0.303)

Q: ¿Qué es la inyección de prompts?
  -> seguridad-001 (sim=0.616)

Q: ¿Para qué sirve el top-k en RAG?
  -> rag-002 (sim=0.503)

Q: ¿Cómo se hace un backup de la base de datos?
  -> apis-001 (sim=0.171)



In [28]:
consultas_de_prueba = [
    "¿Dónde van los parámetros de inferencia cuando están cansados?",
]

for q in consultas_de_prueba:
    vec_q = embed_bedrock([q], input_type="search_query")[0]
    top1_id, top1_score = buscar_fuerza_bruta(vec_q, vectores, k=1)[0]
    print(f"Q: {q}\n  -> {top1_id} (sim={top1_score:.3f})\n")

Q: ¿Dónde van los parámetros de inferencia cuando están cansados?
  -> apis-002 (sim=0.455)



**🏠 Practicá**: para cada consulta de arriba, juzgá a ojo si el resultado top-1 es realmente la mejor respuesta posible dentro del corpus de 6 chunks — anotá cuáles te convencen y cuáles no, y por qué. Esto es, literalmente, el punto de partida informal de lo que la Clase 8 formaliza con métricas.

## 10. Laboratorio: buscador semántico

**🏠 Tarea de la semana — entregar/completar antes de la Clase 6.**

**Checkpoint del proyecto integrador (Clase 5)**: índice vectorial construido sobre el corpus real del TP (no el de juguete de hoy) y buscador semántico funcionando end-to-end.

### Consigna

1. Tomá los chunks + embeddings que generaste en el laboratorio de la Clase 4 (`pipeline_ingesta`).
2. Cargalos en una colección de ChromaDB persistente, con al menos un campo de metadata útil para filtrar (ej. `documento_id`, o una categoría si tu corpus la tiene).
3. Completá la función `buscar` de abajo: dado un texto de consulta, la función embebe la consulta (con `input_type="search_query"` si tu proveedor lo soporta) y devuelve el top-k de la colección.
4. Agregale el filtro de similitud mínima de la practicá de la sección 7 — que `buscar` no devuelva resultados por debajo de un umbral configurable.
5. Corré las 4 consultas de prueba de la sección 9 (o unas propias, relevantes a tu corpus real) contra tu índice real, y evaluá manualmente los resultados.

### Criterio de éxito

- `buscar("una pregunta real sobre tu corpus", k=3)` devuelve chunks razonablemente relevantes, con su `chunk_id` y metadata.
- Una consulta deliberadamente fuera de dominio devuelve pocos o ningún resultado (gracias al umbral), no 3 resultados forzados de baja similitud.
- El índice persiste: si reiniciás el kernel y volvés a abrir la colección, los datos siguen ahí sin re-embeber nada.

In [ ]:
def buscar(consulta: str, coleccion, k: int = 3, umbral_similitud: float = 0.0, where: dict = None) -> list[dict]:
    """Busca los k chunks más relevantes para `consulta` en `coleccion`, descartando los
    que estén por debajo de `umbral_similitud`. `where` permite filtrar por metadata."""
    # TODO: embeber la consulta (ojo con input_type="search_query" si tu proveedor lo distingue)
    # TODO: consultar la colección (query_embeddings=..., n_results=k, where=where)
    # TODO: convertir distancia -> similitud si tu colección usa distancia, y filtrar por umbral_similitud
    # TODO: devolver una lista de dicts con al menos {"chunk_id", "texto", "similitud", "metadata"}
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# resultados = buscar("tu pregunta de prueba", coleccion, k=3, umbral_similitud=0.3)
# for r in resultados:
#     print(r)

## Cierre

- Hoy cerramos el pipeline de recuperación: de un corpus crudo (Clase 4) a un buscador semántico funcional con metadata, persistencia y una noción de "esto no es relevante".
- Próxima clase: Arquitectura RAG — conectamos este buscador con un modelo generador para producir respuestas sustentadas en la evidencia recuperada.
- Antes de la próxima clase: dejar el laboratorio de la sección 10 funcionando sobre el corpus real del TP.